### Import required modules and set environment variables

In [1]:
import os
import requests
from datetime import datetime
import glob
import logging

In [2]:
OUTPUT_FOLDER = r'..\00_storage_raw\yellow_taxi'
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
START_YEAR = 2023
START_MONTH = 1
HEADERS = {"User-Agent": "Mozilla/5.0","Accept": "*/*"}


### Setup Logging

In [3]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "00_ingestion_to_raw.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

### Scan the folder to get the latest month

In [4]:
def get_latest_downloaded_month(folder):

    if not os.path.exists(folder):
        os.makedirs(folder, exist_ok=True)
        return None

    files = glob.glob(f"{folder}/year=*/yellow_tripdata_*.parquet")

    max_date = None

    for file_path in files:
        file_name = os.path.basename(file_path)

        try:
            date_part = file_name.replace("yellow_tripdata_", "").replace(".parquet", "")
            curr_date = datetime.strptime(date_part, "%Y-%m")

            if max_date is None or curr_date > max_date:
                max_date = curr_date

        except ValueError:
            continue

    return max_date


In [5]:
def add_months(source_date, months):
    new_month = source_date.month - 1 + months
    year = source_date.year + new_month // 12
    month = new_month % 12 + 1
    return datetime(year, month, 1)

In [6]:
def incremental_load():

    logging.info("Starting yellow taxi ingestion")
    last_date = get_latest_downloaded_month(OUTPUT_FOLDER)
    
    if last_date:
        next_date = add_months(last_date, 1)
        logging.info(f"Data found till {last_date.strftime('%Y-%m')}. Updates starting from {next_date.strftime('%Y-%m')}")
    else:
        next_date = datetime(START_YEAR, START_MONTH, 1)
        logging.info(f"No existing data found. Pulling batch from {next_date.strftime('%Y-%m')}")

    while True:
        year = next_date.strftime('%Y')
        month = next_date.strftime('%m')
        file_name = f"yellow_tripdata_{year}-{month}.parquet"
        url = f"{BASE_URL}/{file_name}"

        year_folder = os.path.join(OUTPUT_FOLDER, f"year={year}")
        os.makedirs(year_folder, exist_ok=True)
        save_path = os.path.join(year_folder, file_name)

        try:
            logging.info(f"Downloading {file_name}")
            response = requests.get(url, headers = HEADERS, stream=True, timeout = 30)

            # Stop if file doesn't exist
            if response.status_code == 404:
                logging.error("No more files available. Stopping.")
                break

            # Stop if blocked
            if response.status_code == 403:
                logging.error(f"Access denied / blocked for {year}-{month}. Stopping safely.")
                break

            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

            logging.info(f"Successfully ingested {file_name}")

            next_date = add_months(next_date, 1)

        except Exception as e:
            logging.exception(f"Exception: {e}")
            break


In [7]:
if __name__ == "__main__":
    incremental_load()